In [1]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  17.6M      0 --:--:-- --:--:-- --:--:-- 17.7M


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.0 MB/s eta 0:00:00


In [4]:
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics

from IPython.display import Video, display
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
VIDEO_PATH = Path("/content/development.mp4")

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_PATH = OUTPUT_DIR / "bytetrack_baseline.mp4"

TRACKS_CSV_PATH = OUTPUT_DIR / "tracks.csv"

MODEL_NAME = "yolo26n.pt"
TRACKER_CONFIG = "bytetrack.yaml"

IMAGE_SIZE = 640
DETECTION_CONFIDENCE = 0.10
NMS_IOU_THRESHOLD  = 0.70

TARGET_CLASS_NAMES = {
    "person",
    "bicycle",
    "car",
}

DEVICE = torch.device(0 if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

Device: cuda:0
GPU: Tesla T4


**Model and vehicle class IDs**

In [ ]:
class_lookup_model = YOLO(MODEL_NAME)

model_names = class_lookup_model.names

if isinstance(model_names, dict):
    class_id_to_name = {
        int(class_id): str(class_name)
        for class_id, class_name in model_names.items()
    }
else:
    class_id_to_name = {
        class_id: str(class_name)
        for class_id, class_name in enumerate(model_names)
    }

TARGET_CLASS_IDS = sorted(
    class_id 
    for class_id, class_name in class_id_to_name.items()
    if class_name.lower() in TARGET_CLASS_NAMES
)

selected_classes_df = pd.DataFrame([
    {
        "class_id": class_id,
        "class_name": class_id_to_name[class_id]
    }
    for class_id in TARGET_CLASS_IDS
])

display(selected_classes_df)
print("Selected class IDs:", TARGET_CLASS_IDS)

,class_id,class_name
0,0,person
1,1,bicycle
2,2,car


Selected class IDs: [0, 1, 2]
